# Imports

In [1]:
import pandas as pd
import yfinance as yf
import warnings
warnings.filterwarnings("ignore")

# Load Membership Panel

In [2]:
panel = pd.read_csv(r"..\..\Data\Main\membership_panel_clean.csv")
print(f"Tickers to download: {len(panel)}")
print(panel.head())

Tickers to download: 606
  yf_ticker                         name  first_seen   last_seen  \
0         A     Agilent Technologies Inc  2020-01-02  2025-12-31   
1       AAL  American Airlines Group Inc  2020-01-02  2024-07-01   
2       AAP       Advance Auto Parts Inc  2020-01-02  2023-07-03   
3      AAPL                    Apple Inc  2020-01-02  2025-12-31   
4      ABBV                   AbbVie Inc  2020-01-02  2025-12-31   

   quarters_present  
0                24  
1                19  
2                15  
3                24  
4                24  


# Download Daily Prices from yfinance

Each ticker is downloaded from its first_seen date to last_seen + 1 day (to ensure full coverage at panel edges). Tickers that fail are typically acquired or delisted companies whose data has been removed from Yahoo Finance.

In [5]:
import csv

tickers = panel['yf_ticker'].tolist()
success = []
failed = []
first_ticker = True

output_path = r"..\..\Data\Main\sp500_prices_2020_2025.csv"

for i, t in enumerate(tickers):
    row = panel[panel['yf_ticker'] == t].iloc[0]
    start = pd.to_datetime(row['first_seen'])
    end = pd.to_datetime(row['last_seen']) + pd.DateOffset(days=1)

    print(f"{i+1}/{len(tickers)} — {t} ({start.date()} to {end.date()})", end=" ... ")
    try:
        data = yf.download(t, start=start, end=end, progress=False)
        close = data['Close'].dropna()
        if len(close) == 0:
            failed.append(t)
            print("FAILED")
        else:
            first = close.index[0].strftime('%Y-%m-%d')
            last = close.index[-1].strftime('%Y-%m-%d')
            success.append({'ticker': t, 'first_date': first, 'last_date': last, 'days': len(close)})
            
            # Flatten columns and add ticker
            data = data.reset_index()
            data.columns = [c[0] if isinstance(c, tuple) else c for c in data.columns]
            data.rename(columns={'Date': 'date', 'Price': 'date'}, inplace=True)
            data['ticker'] = t
            
            # Write header only for first ticker
            data.to_csv(output_path, mode='a' if not first_ticker else 'w', 
                       header=first_ticker, index=False)
            first_ticker = False
            
            print(f"OK ({len(close)} days)")
    except:
        failed.append(t)
        print("FAILED")

print(f"\nSuccess: {len(success)}")
print(f"Failed: {len(failed)}")
print(f"Failed tickers: {failed}")

success_df = pd.DataFrame(success)

1/606 — A (2020-01-02 to 2026-01-01) ... OK (1508 days)
2/606 — AAL (2020-01-02 to 2024-07-02) ... OK (1131 days)
3/606 — AAP (2020-01-02 to 2023-07-04) ... OK (881 days)
4/606 — AAPL (2020-01-02 to 2026-01-01) ... OK (1508 days)
5/606 — ABBV (2020-01-02 to 2026-01-01) ... OK (1508 days)
6/606 — ABMD (2020-01-02 to 2022-10-04) ... 

$ABMD: possibly delisted; no timezone found

1 Failed download:
['ABMD']: possibly delisted; no timezone found


FAILED
7/606 — ABNB (2023-10-02 to 2026-01-01) ... OK (565 days)
8/606 — ABT (2020-01-02 to 2026-01-01) ... OK (1508 days)
9/606 — ACGL (2023-01-03 to 2026-01-01) ... OK (752 days)
10/606 — ACN (2020-01-02 to 2026-01-01) ... OK (1508 days)
11/606 — ADBE (2020-01-02 to 2026-01-01) ... OK (1508 days)
12/606 — ADI (2020-01-02 to 2026-01-01) ... OK (1508 days)
13/606 — ADM (2020-01-02 to 2026-01-01) ... OK (1508 days)
14/606 — ADP (2020-01-02 to 2026-01-01) ... OK (1508 days)
15/606 — ADSK (2020-01-02 to 2026-01-01) ... OK (1508 days)
16/606 — AEE (2020-01-02 to 2026-01-01) ... OK (1508 days)
17/606 — AEP (2020-01-02 to 2026-01-01) ... OK (1508 days)
18/606 — AES (2020-01-02 to 2026-01-01) ... OK (1508 days)
19/606 — AFL (2020-01-02 to 2026-01-01) ... OK (1508 days)
20/606 — AGN (2020-01-02 to 2020-04-02) ... 

$AGN: possibly delisted; no timezone found

1 Failed download:
['AGN']: possibly delisted; no timezone found


FAILED
21/606 — AIG (2020-01-02 to 2026-01-01) ... OK (1508 days)
22/606 — AIV (2020-01-02 to 2020-10-02) ... OK (190 days)
23/606 — AIZ (2020-01-02 to 2026-01-01) ... OK (1508 days)
24/606 — AJG (2020-01-02 to 2026-01-01) ... OK (1508 days)
25/606 — AKAM (2020-01-02 to 2026-01-01) ... OK (1508 days)
26/606 — ALB (2020-01-02 to 2026-01-01) ... OK (1508 days)
27/606 — ALGN (2020-01-02 to 2026-01-01) ... OK (1508 days)
28/606 — ALK (2020-01-02 to 2023-10-03) ... OK (944 days)
29/606 — ALL (2020-01-02 to 2026-01-01) ... OK (1508 days)
30/606 — ALLE (2020-01-02 to 2026-01-01) ... OK (1508 days)
31/606 — ALXN (2020-01-02 to 2021-07-02) ... 

...

FAILED
588/606 — WSM (2025-04-01 to 2026-01-01) ... OK (190 days)
589/606 — WST (2020-07-01 to 2026-01-01) ... OK (1383 days)
590/606 — WTW (2020-01-02 to 2026-01-01) ... OK (1508 days)
591/606 — WU (2020-01-02 to 2021-10-02) ... OK (442 days)
592/606 — WY (2020-01-02 to 2026-01-01) ... OK (1508 days)
593/606 — WYNN (2020-01-02 to 2026-01-01) ... OK (1508 days)
594/606 — XEC (2020-01-02 to 2020-01-03) ... 

$XEC: possibly delisted; no timezone found

1 Failed download:
['XEC']: possibly delisted; no timezone found


FAILED
595/606 — XEL (2020-01-02 to 2026-01-01) ... OK (1508 days)
596/606 — XLNX (2020-01-02 to 2022-01-04) ... 

$XLNX: possibly delisted; no timezone found

1 Failed download:
['XLNX']: possibly delisted; no timezone found


FAILED
597/606 — XOM (2020-01-02 to 2026-01-01) ... OK (1508 days)
598/606 — XRAY (2020-01-02 to 2024-04-02) ... OK (1068 days)
599/606 — XRX (2020-01-02 to 2021-01-05) ... OK (254 days)
600/606 — XYL (2020-01-02 to 2026-01-01) ... OK (1508 days)
601/606 — XYZ (2025-10-01 to 2026-01-01) ... OK (64 days)
602/606 — YUM (2020-01-02 to 2026-01-01) ... OK (1508 days)
603/606 — ZBH (2020-01-02 to 2026-01-01) ... OK (1508 days)
604/606 — ZBRA (2020-01-02 to 2026-01-01) ... OK (1508 days)
605/606 — ZION (2020-01-02 to 2024-01-03) ... OK (1007 days)
606/606 — ZTS (2020-01-02 to 2026-01-01) ... OK (1508 days)

Success: 560
Failed: 46
Failed tickers: ['ABMD', 'AGN', 'ALXN', 'ANSS', 'ARNC', 'ATVI', 'CERN', 'CMA', 'CTLT', 'CTXS', 'CXO', 'DAY', 'DFS', 'DISCA', 'DISCK', 'DISH', 'DRE', 'ETFC', 'FLIR', 'HBI', 'HES', 'HFC', 'IPG', 'JNPR', 'JWN', 'K', 'KLG', 'KSU', 'MRO', 'MXIM', 'MYL', 'NBL', 'NLSN', 'PARA', 'PBCT', 'PXD', 'RTN', 'SBNY', 'SIVBQ', 'TWTR', 'VAR', 'WBA', 'WCG', 'WRK', 'XEC', 'XLNX']


In [6]:
# Verify the saved file
prices = pd.read_csv(r"..\..\Data\Main\sp500_prices_2020_2025.csv")
print(f"Shape: {prices.shape}")
print(f"Tickers: {prices['ticker'].nunique()}")
print(f"Date range: {prices['date'].min()} to {prices['date'].max()}")
print(f"Columns: {list(prices.columns)}")

success_df.to_csv(r"..\..\Data\Main\price_download_log.csv", index=False)
failed_df = pd.DataFrame({'ticker': failed})
failed_df.to_csv(r"..\..\Data\Main\price_download_failed.csv", index=False)

print(f"\nSaved:")
print(f"  sp500_prices_2020_2025.csv — {len(prices)} rows, {prices['ticker'].nunique()} tickers")
print(f"  price_download_log.csv — {len(success_df)} successful tickers")
print(f"  price_download_failed.csv — {len(failed_df)} failed tickers")

Shape: (722756, 7)
Tickers: 560
Date range: 2020-01-02 to 2025-12-31
Columns: ['date', 'Close', 'High', 'Low', 'Open', 'Volume', 'ticker']

Saved:
  sp500_prices_2020_2025.csv — 722756 rows, 560 tickers
  price_download_log.csv — 560 successful tickers
  price_download_failed.csv — 46 failed tickers


# Checking Failed Ticker Quarter Duration

In [7]:
# Checking how long these tickers were in the index
failed_panel = panel[panel['yf_ticker'].isin(failed)]
print(f"Failed tickers: {len(failed)}")
print(f"\nMembership duration of failed tickers:")
print(failed_panel[['yf_ticker', 'name', 'first_seen', 'last_seen', 'quarters_present']].to_string(index=False))

Failed tickers: 46

Membership duration of failed tickers:
yf_ticker                           name first_seen  last_seen  quarters_present
     ABMD                    ABIOMED Inc 2020-01-02 2022-10-03                12
      AGN              Allergan UnLtd Co 2020-01-02 2020-04-01                 2
     ALXN Alexion Pharmaceuticals Inc/MA 2020-01-02 2021-07-01                 7
     ANSS                      ANSYS Inc 2020-01-02 2025-07-01                23
     ARNC                   Arconic Corp 2020-04-01 2020-04-01                 1
     ATVI        Activision Blizzard Inc 2020-01-02 2023-10-02                16
     CERN                    Cerner Corp 2020-01-02 2022-04-01                10
      CMA                   Comerica Inc 2020-01-02 2024-04-01                18
     CTLT                   Catalent Inc 2020-10-01 2024-10-01                17
     CTXS             Citrix Systems Inc 2020-01-02 2022-07-01                11
      CXO           Concho Resources Inc 2020-01-0

# Categorise Failed Tickers by Membership Duration

In [8]:
failed_panel = panel[panel['yf_ticker'].isin(failed)].copy()

short = failed_panel[failed_panel['quarters_present'] <= 4]
medium = failed_panel[(failed_panel['quarters_present'] > 4) & (failed_panel['quarters_present'] <= 12)]
long_term = failed_panel[failed_panel['quarters_present'] > 12]

print(f"Short membership (1-4 quarters): {len(short)} tickers")
print(short[['yf_ticker', 'name', 'quarters_present']].to_string(index=False))

print(f"\nMedium membership (5-12 quarters): {len(medium)} tickers")
print(medium[['yf_ticker', 'name', 'quarters_present']].to_string(index=False))

print(f"\nLong membership (13+ quarters): {len(long_term)} tickers")
print(long_term[['yf_ticker', 'name', 'quarters_present']].to_string(index=False))

Short membership (1-4 quarters): 10 tickers
yf_ticker                        name  quarters_present
      AGN           Allergan UnLtd Co                 2
     ARNC                Arconic Corp                 1
     ETFC      E*TRADE Financial Corp                 4
      JWN               Nordstrom Inc                 2
      KLG               WK Kellogg Co                 1
      MYL                    Mylan NV                 4
      NBL            Noble Energy Inc                 4
      RTN                 Raytheon Co                 2
      WCG   WellCare Health Plans Inc                 1
      XEC Coterra Energy Operating Co                 1

Medium membership (5-12 quarters): 19 tickers
yf_ticker                           name  quarters_present
     ABMD                    ABIOMED Inc                12
     ALXN Alexion Pharmaceuticals Inc/MA                 7
     CERN                    Cerner Corp                10
     CTXS             Citrix Systems Inc                1

# Quantify Data Loss from Excluded Tickers

In [9]:
success_panel = panel[panel['yf_ticker'].isin([s['ticker'] for s in success])]

failed_panel['est_trading_days'] = failed_panel['quarters_present'] * 63
total_possible = panel['quarters_present'].sum() * 63
total_lost = failed_panel['est_trading_days'].sum()

print(f"Total tickers in panel: {len(panel)}")
print(f"Successfully downloaded: {len(success)}")
print(f"Failed: {len(failed)} ({len(failed)/len(panel)*100:.1f}%)")
print(f"\nEstimated trading days lost: {total_lost:,} out of {total_possible:,} ({total_lost/total_possible*100:.1f}%)")

Total tickers in panel: 606
Successfully downloaded: 560
Failed: 46 (7.6%)

Estimated trading days lost: 33,012 out of 761,418 (4.3%)


# Investigate Failed Tickers via Yahoo Finance Search API

In [10]:
import requests
import time

print("Investigating each failed ticker via Yahoo Finance search:\n")
results = []

for _, row in failed_panel.iterrows():
    t = row['yf_ticker']
    name = row['name']
    
    try:
        url = f"https://query2.finance.yahoo.com/v1/finance/search?q={name}&quotesCount=3"
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers)
        quotes = response.json().get('quotes', [])
        
        if quotes:
            top = quotes[0]
            symbol = top['symbol']
            match_name = top.get('longname', top.get('shortname', 'N/A'))
            print(f"{t:>6} ({name})")
            print(f"       → {symbol} ({match_name})")
        else:
            print(f"{t:>6} ({name})")
            print(f"       → No results — likely fully delisted or taken private")
        
        results.append({
            'ticker': t,
            'name': name,
            'quarters_present': row['quarters_present'],
            'last_seen': row['last_seen'],
            'yf_search_result': quotes[0]['symbol'] if quotes else 'NONE',
            'yf_search_name': quotes[0].get('longname', 'N/A') if quotes else 'N/A'
        })
    except Exception as e:
        print(f"{t:>6} ({name}) → Error: {e}")
        results.append({
            'ticker': t, 'name': name,
            'quarters_present': row['quarters_present'],
            'last_seen': row['last_seen'],
            'yf_search_result': 'ERROR', 'yf_search_name': str(e)
        })
    
    time.sleep(0.5)

results_df = pd.DataFrame(results)
results_df.to_csv(r"..\..\Data\Main\failed_tickers_investigation.csv", index=False)
print(f"\nSaved to failed_tickers_investigation.csv")

Investigating each failed ticker via Yahoo Finance search:

  ABMD (ABIOMED Inc)
       → No results — likely fully delisted or taken private
   AGN (Allergan UnLtd Co)
       → No results — likely fully delisted or taken private
  ALXN (Alexion Pharmaceuticals Inc/MA)
       → No results — likely fully delisted or taken private
  ANSS (ANSYS Inc)
       → No results — likely fully delisted or taken private
  ARNC (Arconic Corp)
       → No results — likely fully delisted or taken private
  ATVI (Activision Blizzard Inc)
       → No results — likely fully delisted or taken private
  CERN (Cerner Corp)
       → No results — likely fully delisted or taken private
   CMA (Comerica Inc)
       → No results — likely fully delisted or taken private
  CTLT (Catalent Inc)
       → No results — likely fully delisted or taken private
  CTXS (Citrix Systems Inc)
       → No results — likely fully delisted or taken private
   CXO (Concho Resources Inc)
       → No results — likely fully delisted o

# Review Investigation Results

In [11]:
no_results = results_df[results_df['yf_search_result'] == 'NONE']
has_results = results_df[results_df['yf_search_result'] != 'NONE']

print(f"Tickers fully removed from Yahoo Finance: {len(no_results)} out of {len(results_df)}")
print(f"\nTickers with search results ({len(has_results)}):")
for _, row in has_results.iterrows():
    print(f"  {row['ticker']} → {row['yf_search_result']} ({row['yf_search_name']})")

Tickers fully removed from Yahoo Finance: 42 out of 46

Tickers with search results (4):
  DISCA → WBD (Warner Bros. Discovery, Inc.)
  DISCK → WBD (Warner Bros. Discovery, Inc.)
  SBNY → SBNYL (Signature Bank)
  WRK → WEST (Westrock Coffee Company)


# Investigation Findings

**Yahoo Finance search results:**
- DISCA/DISCK → WBD (Warner Bros Discovery): Discovery Inc share classes that merged into WBD, which is already in our panel. No data loss.
- SBNY → SBNYL: Only a residual security remains. Common stock data was removed after the bank collapsed in March 2023.
- WRK → WEST: False match. Westrock Coffee is a different company from WestRock Co (packaging), which merged with Smurfit Kappa in July 2024.

**All remaining 42 tickers returned no results — fully removed from Yahoo Finance.**
46 tickers returned no data from Yahoo Finance. All 46 display "possibly delisted" errors from yfinance, indicating their securities have been removed from Yahoo Finance's database.Three (DAY, IPG, K) were verified as recent acquisitions via Bloomberg Terminal DES command:
- DAY (Dayforce) — acquired by Thoma Bravo, completed Feb 2026
- IPG (Interpublic) — acquired by Omnicom Group, completed Nov 2025
- K (Kellanova) — acquired by Mars Inc, completed Dec 2025

The remaining 39 are consistent with companies acquired, taken private, or otherwise delisted during the study period. Total estimated data loss is 4.3% of trading days — insufficient to introduce systematic bias.

All 46 excluded from analysis. Remaining: 560 tickers with price data.